In [2]:
!pip install "transformers[torch]"

In [3]:
import pandas as pd
from transformers import T5Tokenizer , Trainer, TrainingArguments, T5ForConditionalGeneration

In [19]:
from google.colab import files

# Run this cell to upload 'samsum-train.csv' from your desktop
uploaded = files.upload()

KeyboardInterrupt: 

In [10]:
train_data = pd.read_csv('samsum-train.csv')
val_data = pd.read_csv('samsum-validation.csv')


In [8]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."




Random sampling

In [11]:
train_data = train_data.sample(n=4000, random_state = 42).reset_index(drop = True)
val_data = val_data.sample(n = 500, random_state = 42).reset_index(drop=True)

In [13]:
train_data.shape

(4000, 3)

DATA PREPROCESSING

In [14]:
import re

def clean_data(text):
  text = re.sub(r"\r\n", " ", text) #lines
  text = re.sub(r"\s+", " ", text) #spaces
  text = re.sub(r"<.*?>", " ", text) #html tags
  return text

In [15]:
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data)


In [17]:
train_data["dialogue"][0]

"Violet: hi! i came across this Austin's article and i thought that you might find it interesting Violet:   Claire: Hi! :) Thanks, but I've already read it. :) Claire: But thanks for thinking about me :)"

Tokenization

In [22]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [29]:
def tokenize(data):
  inputs = tokenizer(data["dialogue"], padding ="max_length" , max_length = 512, truncation = True)
  targets = tokenizer(data["summary"], padding = "max_length", max_length = 150, truncation= True)

  inputs["labels"] = targets["input_ids"]
  return inputs

In [30]:
train_dataset = train_data.apply(tokenize, axis = 1).tolist()
val_dataset = val_data.apply(tokenize, axis = 1).tolist()

In [31]:
train_dataset[0]

{'input_ids': [28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [33]:
len(train_dataset[0]["labels"])

150

Working with Model

In [35]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [37]:
import torch

if torch.backends.mps.is_available():
   device = torch.device("mps")
elif torch.cuda.is_available():
   device = torch.device("cuda")
else:
     device = torch.device("cpu")
print("device: ", device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [38]:
# Training Arguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs =6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500,

)

In [39]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [41]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.358592,0.351576
2,0.353880,0.349615
3,0.344138,0.347892
4,0.337113,0.346466
5,0.331678,0.347042
6,0.328784,0.346305


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.3423641866048177, metrics={'train_runtime': 1284.5502, 'train_samples_per_second': 18.684, 'train_steps_per_second': 2.335, 'total_flos': 3248203235328000.0, 'train_loss': 0.3423641866048177, 'epoch': 6.0})

In [42]:
#Save the Model
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [51]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Test the Core Logic

In [52]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue) #data cleaning

  inputs = tokenizer(
      dialogue,
      padding = "max_length",
      max_length = 512,
      truncation = True,
      return_tensors = "pt"
  )

  #generate summary => token id generation
  targets = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length = 150,
      num_beams = 4,
      early_stopping = True
  )

  # decode the output
  summary = tokenizer.decode(targets[0], skip_special_tokens = True)

  return summary


In [53]:
test_dialogue = """
Reporter: Thanks for joining me, Dr. Rao. Cities keep breaking heat records. What's driving that?

Dr. Rao: Two main things. Global warming raises the baseline temperature, and the "urban heat island" effect adds to it. Concrete, asphalt, and dark roofs absorb heat all day and release it at night, so cities can run five to ten degrees hotter than nearby rural areas.

Reporter: Why does the nighttime part matter so much?

Dr. Rao: Because bodies need to cool down overnight. When temperatures stay high after dark, people don't recover, and that's when heat-related hospital visits climb, especially among older adults and people without air conditioning.

Reporter: What can cities realistically do about it?

Dr. Rao: Planting trees is the most effective low-cost option. A mature tree canopy can lower surface temperatures by several degrees through shade and evapotranspiration. Reflective "cool roofs" and light-colored pavement help too, but trees also improve air quality and mental health.

Reporter: So why isn't every city doing this already?

Dr. Rao: Cost, space, and time. Saplings take decades to provide full shade, and wealthier neighborhoods often have far more tree cover than low-income ones. Fixing that inequity needs targeted funding and community involvement, not just planting numbers.

Reporter: If you could give city planners one piece of advice, what would it be?

Dr. Rao: Start with the hottest, most vulnerable neighborhoods first, and protect the trees you already have. Preserving existing canopy is cheaper and faster than replacing it.

Reporter: Thank you, Dr. Rao.

Dr. Rao: My pleasure.
"""

sumary = summarize_dialogue(test_dialogue)
print("summary: ", sumary)

summary:  Dr. Rao has joined him. Cities keep breaking heat records. Cities can run five to ten degrees hotter than nearby rural areas.


In [54]:
!zip -r saved_summary_model.zip saved_summary_model

  adding: saved_summary_model/ (stored 0%)
  adding: saved_summary_model/tokenizer_config.json (deflated 82%)
  adding: saved_summary_model/generation_config.json (deflated 29%)
  adding: saved_summary_model/tokenizer.json (deflated 75%)
  adding: saved_summary_model/model.safetensors (deflated 9%)
  adding: saved_summary_model/config.json (deflated 63%)
